In [16]:
import pandas as pd
import psycopg2
from psycopg2 import sql


In [17]:
# Load bank information
df_banks = pd.read_csv("../data/processed/bank_reviews_cleaned.csv")

# Load review information
df_reviews = pd.read_csv("../data/processed/reviews_for_database.csv")

# Quick check
print(df_banks.head())
print(df_reviews.head())


                                              review  rating        date  \
0                                          very good       5  2025-11-25   
1           most of the time is not working properly       1  2025-11-25   
2                                       good service       5  2025-11-25   
3                                     not use for me       3  2025-11-23   
4  It keeps notifying me to disable developer opt...       1  2025-11-22   

                bank       source                             review_id  \
0  Bank of Abyssinia  Google Play  7ef21cf6-d226-4370-ab96-01c909dbc58d   
1  Bank of Abyssinia  Google Play  896ee9aa-a483-4b1f-b73c-0a26c4b54790   
2  Bank of Abyssinia  Google Play  15c3586b-e672-48db-b3c0-09508375763f   
3  Bank of Abyssinia  Google Play  6f7113d8-180e-4f3d-83d9-fbe55f9edd69   
4  Bank of Abyssinia  Google Play  d36fbdcb-b57e-4384-8e5b-ae549e25b33e   

                                         review_text  review_year  \
0                      

Connect to PostgreSQL

In [20]:
# Connect to default 'postgres' database to create our new DB
conn = psycopg2.connect(
    host="localhost",
    database="postgres",  # default DB
    user="postgres"       # your PostgreSQL user
    # password=None if no password
)

cur = conn.cursor()


Create Database

In [22]:
import psycopg2

# Connect to default 'postgres' DB
conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres"
)

# Enable autocommit
conn.autocommit = True

cur = conn.cursor()

# Drop old DB if exists, then create new
cur.execute("DROP DATABASE IF EXISTS bank_reviews;")
cur.execute("CREATE DATABASE bank_reviews;")

cur.close()
conn.close()

print("Database created successfully!")


Database created successfully!


Connect to the new database

In [5]:
import psycopg2

# Connect to the newly created database
conn = psycopg2.connect(
    host="localhost",
    database="bank_reviews",
    user="postgres"
)

cur = conn.cursor()
print("Connected to bank_reviews database")


Connected to bank_reviews database


Create the tables

In [9]:
df_banks = pd.read_csv("data/processed/bank_reviews_cleaned.csv")
df_reviews = pd.read_csv("data/processed/reviews_for_database.csv")

In [12]:
df_banks.info()
df_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   review        1200 non-null   object
 1   rating        1200 non-null   int64 
 2   date          1200 non-null   object
 3   bank          1200 non-null   object
 4   source        1200 non-null   object
 5   review_id     1200 non-null   object
 6   review_text   1200 non-null   object
 7   review_year   1200 non-null   int64 
 8   review_month  1200 non-null   int64 
 9   bank_code     1200 non-null   object
 10  user_name     1200 non-null   object
 11  thumbs_up     1200 non-null   int64 
 12  text_length   1200 non-null   int64 
 13  word_count    1200 non-null   int64 
 14  app_version   1200 non-null   object
dtypes: int64(6), object(9)
memory usage: 140.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 5 columns):
 #   Column          

In [13]:
df_banks.info()
df_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   review        1200 non-null   object
 1   rating        1200 non-null   int64 
 2   date          1200 non-null   object
 3   bank          1200 non-null   object
 4   source        1200 non-null   object
 5   review_id     1200 non-null   object
 6   review_text   1200 non-null   object
 7   review_year   1200 non-null   int64 
 8   review_month  1200 non-null   int64 
 9   bank_code     1200 non-null   object
 10  user_name     1200 non-null   object
 11  thumbs_up     1200 non-null   int64 
 12  text_length   1200 non-null   int64 
 13  word_count    1200 non-null   int64 
 14  app_version   1200 non-null   object
dtypes: int64(6), object(9)
memory usage: 140.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 5 columns):
 #   Column          

Review the data frame

In [33]:
print(df_banks.columns)
print(df_reviews.columns)

Index(['review', 'rating', 'date', 'bank', 'source', 'review_id',
       'review_text', 'review_year', 'review_month', 'bank_code', 'user_name',
       'thumbs_up', 'text_length', 'word_count', 'app_version'],
      dtype='object')
Index(['review_id', 'review_text', 'sentiment_label', 'sentiment_score',
       'identified_theme(s)'],
      dtype='object')


In [ ]:
Prepare banks DataFrame

In [ ]:
import pandas as pd

# Load cleaned data
df_banks = pd.read_csv("data/processed/bank_reviews_cleaned.csv")
df_reviews = pd.read_csv("data/processed/reviews_for_database.csv")

# Prepare banks DataFrame
df_banks_db = df_banks[['bank_code', 'bank']].drop_duplicates()
df_banks_db = df_banks_db.rename(columns={'bank_code': 'bank_id', 'bank': 'bank_name'})
df_banks_db['app_name'] = None  # fill optional app_name column

# Prepare reviews DataFrame (optional: convert date column to datetime)
df_reviews['review_date'] = pd.to_datetime(df_reviews['review_date'], errors='coerce')


Insert bank data

In [ ]:
# Insert banks data
for _, row in df_banks.iterrows():
    cur.execute("""
        INSERT INTO banks (bank_id, bank_name, app_name)
        VALUES (%s, %s, %s)
        ON CONFLICT (bank_id) DO NOTHING;
    """, (row['bank_code'], row['bank'], row.get('app_version', None)))

conn.commit()

# Insert reviews data
for _, row in df_reviews.iterrows():
    cur.execute("""
        INSERT INTO reviews (review_id, review_text, sentiment_label, sentiment_score)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (review_id) DO NOTHING;
    """, (row['review_id'], row['review_text'], row['sentiment_label'], row['sentiment_score']))

conn.commit()


Connect to PostgreSQL using psycopg2

In [1]:
import psycopg2

conn = psycopg2.connect(
    dbname="bank_reviews",
    user="postgres",
    host="localhost",
    port="5432"
)
cur = conn.cursor()


Insert Banks and Reviews

In [3]:
import pandas as pd
import psycopg2

# Load cleaned CSVs
df_banks = pd.read_csv("data/processed/bank_reviews_cleaned.csv")
df_reviews = pd.read_csv("data/processed/reviews_for_database.csv")

# Prepare banks DataFrame
df_banks_db = df_banks[['bank_code', 'bank']].drop_duplicates()
df_banks_db = df_banks_db.rename(columns={'bank_code': 'bank_id', 'bank': 'bank_name'})
df_banks_db['app_name'] = None

# Connect to PostgreSQL
conn = psycopg2.connect(
    dbname="bank_reviews",
    user="postgres",
    host="localhost",
    port="5432"
)
cur = conn.cursor()
print("Connected to bank_reviews database!")


Connected to bank_reviews database!


In [4]:
# Insert banks
for _, row in df_banks_db.iterrows():
    cur.execute("""
        INSERT INTO banks (bank_id, bank_name, app_name)
        VALUES (%s, %s, %s)
        ON CONFLICT (bank_id) DO NOTHING;
    """, (row['bank_id'], row['bank_name'], row['app_name']))

conn.commit()
print("Banks data inserted successfully!")

Banks data inserted successfully!


In [7]:
print(df_reviews.columns)

Index(['review_id', 'review_text', 'sentiment_label', 'sentiment_score',
       'identified_theme(s)'],
      dtype='object')


In [9]:
# Load the original reviews CSV that contains 'bank_code'
df_reviews_full = pd.read_csv("data/processed/bank_reviews_cleaned.csv")

# Keep only necessary columns for the reviews table
df_reviews_db = df_reviews_full[['review_id', 'review_text', 'bank_code']].copy()

# Add sentiment info from your processed df_reviews
df_reviews_db = df_reviews_db.merge(
    df_reviews[['review_id', 'sentiment_label', 'sentiment_score']],
    on='review_id',
    how='left'
)

# Rename bank_code to bank_id to match PostgreSQL
df_reviews_db = df_reviews_db.rename(columns={'bank_code': 'bank_id'})

df_reviews_db.head()

,review_id,review_text,bank_id,sentiment_label,sentiment_score
0,7ef21cf6-d226-4370-ab96-01c909dbc58d,very good,BOA,POSITIVE,0.9
1,896ee9aa-a483-4b1f-b73c-0a26c4b54790,most of the time is not working properly,BOA,NEGATIVE,0.2
2,15c3586b-e672-48db-b3c0-09508375763f,good service,BOA,POSITIVE,0.9
3,6f7113d8-180e-4f3d-83d9-fbe55f9edd69,not use for me,BOA,NEUTRAL,0.5
4,d36fbdcb-b57e-4384-8e5b-ae549e25b33e,It keeps notifying me to disable developer opt...,BOA,NEGATIVE,0.2


In [5]:
print(df_reviews.columns)


Index(['review_id', 'review_text', 'sentiment_label', 'sentiment_score',
       'identified_theme(s)'],
      dtype='object')


In [9]:
df_reviews['bank_id'] = 'BOA'

In [10]:
df_reviews['rating'] = None


In [11]:
df_reviews['review_date'] = None


In [12]:
df_reviews['source'] = 'Google Play'

In [13]:
print(df_reviews.columns.tolist())


['review_id', 'review_text', 'sentiment_label', 'sentiment_score', 'identified_theme(s)', 'bank_id', 'rating', 'review_date', 'source']


In [14]:
print(df_reviews.dtypes)


review_id               object
review_text             object
sentiment_label         object
sentiment_score        float64
identified_theme(s)     object
bank_id                 object
rating                  object
review_date             object
source                  object
dtype: object


In [17]:
print(df_reviews.dtypes)
print(df_reviews[['rating', 'review_date']].head())

review_id                      object
review_text                    object
sentiment_label                object
sentiment_score               float64
identified_theme(s)            object
bank_id                        object
rating                        float64
review_date            datetime64[ns]
source                         object
dtype: object
   rating review_date
0     NaN         NaT
1     NaN         NaT
2     NaN         NaT
3     NaN         NaT
4     NaN         NaT


In [18]:
print(df_reviews[['rating', 'review_date']].head(20))


    rating review_date
0      NaN         NaT
1      NaN         NaT
2      NaN         NaT
3      NaN         NaT
4      NaN         NaT
5      NaN         NaT
6      NaN         NaT
7      NaN         NaT
8      NaN         NaT
9      NaN         NaT
10     NaN         NaT
11     NaN         NaT
12     NaN         NaT
13     NaN         NaT
14     NaN         NaT
15     NaN         NaT
16     NaN         NaT
17     NaN         NaT
18     NaN         NaT
19     NaN         NaT


In [19]:
print(df_reviews[['rating', 'review_date']].astype(str).head(20))


   rating review_date
0     nan         NaT
1     nan         NaT
2     nan         NaT
3     nan         NaT
4     nan         NaT
5     nan         NaT
6     nan         NaT
7     nan         NaT
8     nan         NaT
9     nan         NaT
10    nan         NaT
11    nan         NaT
12    nan         NaT
13    nan         NaT
14    nan         NaT
15    nan         NaT
16    nan         NaT
17    nan         NaT
18    nan         NaT
19    nan         NaT


In [20]:
import pandas as pd

# 1️⃣ Load your reviews CSV
df_reviews = pd.read_csv(
    r"D:\Personal\KAIM-10 Academy\Week 2\Project Work\Customer_Exp_Analytics_Fintech Apps\data\processed\reviews_for_database.csv"
)

# 2️⃣ Inspect the first few rows and columns
print(df_reviews.head())
print(df_reviews.columns)
print(df_reviews.dtypes)

# 3️⃣ Ensure all needed columns exist, add missing ones if needed
required_cols = ['review_id', 'review_text', 'sentiment_label', 'sentiment_score', 
                 'identified_theme(s)', 'bank_id', 'rating', 'review_date', 'source']

for col in required_cols:
    if col not in df_reviews.columns:
        df_reviews[col] = None  # Fill missing columns with None

# 4️⃣ Clean numeric columns
# Convert rating to numeric, invalid entries become NaN
df_reviews['rating'] = pd.to_numeric(df_reviews['rating'], errors='coerce')

# 5️⃣ Clean date columns
# Try parsing common date formats
df_reviews['review_date'] = pd.to_datetime(
    df_reviews['review_date'], 
    errors='coerce',  # invalid formats become NaT
    dayfirst=False    # adjust to True if your dates are in DD/MM/YYYY
)

# 6️⃣ Optional: Fill missing bank_id if you have a mapping df_banks_db
# df_banks_db should have columns ['bank_id', 'bank_name']
if 'bank_name' in df_reviews.columns:
    df_reviews = df_reviews.merge(
        df_banks_db[['bank_id', 'bank_name']],
        how='left',
        left_on='bank_name',
        right_on='bank_name'
    )

# 7️⃣ Inspect cleaned data
print(df_reviews[['review_id', 'bank_id', 'rating', 'review_date']].head(20))
print(df_reviews.dtypes)

# ✅ Now df_reviews is ready for PostgreSQL insertion


                              review_id  \
0  7ef21cf6-d226-4370-ab96-01c909dbc58d   
1  896ee9aa-a483-4b1f-b73c-0a26c4b54790   
2  15c3586b-e672-48db-b3c0-09508375763f   
3  6f7113d8-180e-4f3d-83d9-fbe55f9edd69   
4  d36fbdcb-b57e-4384-8e5b-ae549e25b33e   

                                         review_text sentiment_label  \
0                                          very good        POSITIVE   
1           most of the time is not working properly        NEGATIVE   
2                                       good service        POSITIVE   
3                                     not use for me         NEUTRAL   
4  It keeps notifying me to disable developer opt...        NEGATIVE   

   sentiment_score     identified_theme(s)  
0              0.9       Positive Feedback  
1              0.2  App Performance & Bugs  
2              0.9        Customer Support  
3              0.5        General Feedback  
4              0.2        General Feedback  
Index(['review_id', 'review_text', 'se

In [21]:
df_banks_db = pd.DataFrame({
    'bank_id': ['BOA', 'CBE', 'DBE'],  # Example IDs
    'bank_name': ['Bank of Abyssinia', 'Commercial Bank of Ethiopia', 'Dashen Bank']
})


In [26]:
import pandas as pd

df_reviews = pd.read_csv(r"D:\Personal\KAIM-10 Academy\Week 2\Project Work\Customer_Exp_Analytics_Fintech Apps\data\processed\reviews_for_database.csv")
print(df_reviews.columns)


Index(['review_id', 'review_text', 'sentiment_label', 'sentiment_score',
       'identified_theme(s)'],
      dtype='object')


In [27]:
# Add missing columns so they exist for database insertion
for col in ['bank_id', 'rating', 'review_date', 'source']:
    df_reviews[col] = None

In [28]:
import pandas as pd

# rating should be numeric (currently None will stay NaN)
df_reviews['rating'] = pd.to_numeric(df_reviews['rating'], errors='coerce')

# review_date should be datetime (None will stay NaT)
df_reviews['review_date'] = pd.to_datetime(df_reviews['review_date'], errors='coerce').dt.date

In [29]:
df_reviews['bank_id'] = 'BOA'


In [30]:
['review_id', 'review_text', 'sentiment_label', 'sentiment_score',
'identified_theme(s)', 'bank_id', 'rating', 'review_date', 'source']


['review_id',
 'review_text',
 'sentiment_label',
 'sentiment_score',
 'identified_theme(s)',
 'bank_id',
 'rating',
 'review_date',
 'source']

In [31]:
import pandas as pd

# Convert rating to numeric
df_reviews['rating'] = pd.to_numeric(df_reviews['rating'], errors='coerce')

# Convert review_date to datetime
df_reviews['review_date'] = pd.to_datetime(df_reviews['review_date'], errors='coerce').dt.date

# Ensure bank_id is filled (replace 'BOA' with the appropriate bank_id if needed)
df_reviews['bank_id'] = df_reviews['bank_id'].fillna('BOA')

# Optional: fill source if missing
df_reviews['source'] = df_reviews['source'].fillna('Google Play')


In [4]:
import pandas as pd
import psycopg2

# 1️⃣ Load your reviews CSV
df_reviews = pd.read_csv(
    r"D:\Personal\KAIM-10 Academy\Week 2\Project Work\Customer_Exp_Analytics_Fintech Apps\data\processed\reviews_for_database.csv"
)

# Check the columns
print(df_reviews.columns)

# 2️⃣ If you want to add missing columns for database insertion
df_reviews['bank_id'] = None          # or fill with correct IDs later
df_reviews['rating'] = None
df_reviews['review_date'] = None
df_reviews['source'] = None

# 3️⃣ Connect to PostgreSQL
conn = psycopg2.connect(
    dbname="bank_reviews",
    user="postgres",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# 4️⃣ Insert rows into reviews table
for _, row in df_reviews.iterrows():
    cur.execute("""
        INSERT INTO reviews (review_id, bank_id, review_text, rating, review_date,
                             sentiment_label, sentiment_score, source)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (review_id) DO NOTHING;
    """, (
        row['review_id'],
        row['bank_id'],           
        row['review_text'],
        row['rating'],
        row['review_date'],
        row['sentiment_label'],
        row['sentiment_score'],
        row['source']
    ))

conn.commit()
cur.close()
conn.close()
print("All reviews inserted successfully!")

Index(['review_id', 'review_text', 'sentiment_label', 'sentiment_score',
       'identified_theme(s)'],
      dtype='object')
All reviews inserted successfully!


In [5]:
import psycopg2
import pandas as pd

# 1️⃣ Connect to PostgreSQL
conn = psycopg2.connect(
    dbname="bank_reviews",
    user="postgres",
    host="localhost",
    port="5432"
)

# 2️⃣ Create a cursor
cur = conn.cursor()

# 3️⃣ Fetch all banks
cur.execute("SELECT * FROM banks;")
banks_data = cur.fetchall()
banks_columns = [desc[0] for desc in cur.description]
df_banks = pd.DataFrame(banks_data, columns=banks_columns)
print("=== Banks Table ===")
print(df_banks)

# 4️⃣ Fetch all reviews
cur.execute("SELECT * FROM reviews;")
reviews_data = cur.fetchall()
reviews_columns = [desc[0] for desc in cur.description]
df_reviews = pd.DataFrame(reviews_data, columns=reviews_columns)
print("\n=== Reviews Table ===")
print(df_reviews.head(20))  # display first 20 rows for readability

# 5️⃣ Close the connection
cur.close()
conn.close()

=== Banks Table ===
  bank_id                    bank_name app_name
0     BOA            Bank of Abyssinia     None
1     CBE  Commercial Bank of Ethiopia     None
2  DASHEN                  Dashen Bank     None

=== Reviews Table ===
                               review_id bank_id  \
0   7ef21cf6-d226-4370-ab96-01c909dbc58d    None   
1   896ee9aa-a483-4b1f-b73c-0a26c4b54790    None   
2   15c3586b-e672-48db-b3c0-09508375763f    None   
3   6f7113d8-180e-4f3d-83d9-fbe55f9edd69    None   
4   d36fbdcb-b57e-4384-8e5b-ae549e25b33e    None   
5   2d630116-152e-4dbb-bb7a-d7f93bbafbc5    None   
6   b9fa948b-07bc-41f5-9e49-5e612b9a9db6    None   
7   c198f602-e9fb-4b61-ad3a-8efe42269b8d    None   
8   afad642f-39c4-4db0-adbf-5fbe7143f960    None   
9   9fb5fdaa-6172-43c1-ba46-2fcdfcd84c13    None   
10  58ccb5e9-0cf7-413c-8bb4-6515d4863bc1    None   
11  0ed79d57-a54c-4541-9df0-67e1e8c72be3    None   
12  f38caf5b-4580-4a9f-819f-f4cd3d789176    None   
13  6a7a5c64-4056-404d-b37e-60bf1d6ee

In [7]:
# Example: if all reviews in this file are for BOA
df_reviews['bank_id'] = 'BOA'  # assign manually if same bank

In [1]:
# ==============================================
# FINAL FIX: Update all reviews with proper data
# ==============================================

import pandas as pd
import psycopg2
import numpy as np
from datetime import datetime, timedelta

print("🔄 FINAL DATA FIX SCRIPT")
print("="*50)

# 1. Connect to database
conn = psycopg2.connect(
    dbname="bank_reviews",
    user="postgres",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# 2. Get total review count
cur.execute("SELECT COUNT(*) FROM reviews;")
total_reviews = cur.fetchone()[0]
print(f"📊 Total reviews to fix: {total_reviews}")

# 3. Fix bank_id (distribute among 3 banks)
print("🔗 Fixing bank_id assignments...")
bank_ids = ['BOA', 'CBE', 'DASHEN']

cur.execute("SELECT review_id FROM reviews ORDER BY review_id;")
all_review_ids = [row[0] for row in cur.fetchall()]

# Assign banks in rotation
for i, review_id in enumerate(all_review_ids):
    bank_id = bank_ids[i % 3]
    cur.execute(
        "UPDATE reviews SET bank_id = %s WHERE review_id = %s;",
        (bank_id, review_id)
    )
    
    # Show progress
    if (i + 1) % 100 == 0:
        print(f"   Updated {i + 1} reviews...")

# 4. Fix rating (use actual ratings from your CSV)
print("⭐ Fixing ratings...")
df_banks = pd.read_csv("data/processed/bank_reviews_cleaned.csv")

# Create rating mapping from your CSV
rating_map = dict(zip(df_banks['review_id'], df_banks['rating']))

for review_id, rating in rating_map.items():
    cur.execute(
        "UPDATE reviews SET rating = %s WHERE review_id = %s;",
        (float(rating), review_id)
    )

# 5. Fix review_date (use actual dates from your CSV)
print("📅 Fixing review dates...")
date_map = dict(zip(df_banks['review_id'], df_banks['date']))

for review_id, date_str in date_map.items():
    try:
        # Convert to proper date format
        date_obj = pd.to_datetime(date_str).date()
        cur.execute(
            "UPDATE reviews SET review_date = %s WHERE review_id = %s;",
            (date_obj, review_id)
        )
    except:
        # Use today's date if parsing fails
        cur.execute(
            "UPDATE reviews SET review_date = CURRENT_DATE WHERE review_id = %s;",
            (review_id,)
        )

# 6. Fix source
print("🌐 Fixing sources...")
source_map = dict(zip(df_banks['review_id'], df_banks['source']))

for review_id, source in source_map.items():
    cur.execute(
        "UPDATE reviews SET source = %s WHERE review_id = %s;",
        (source, review_id)
    )

# 7. Commit all changes
conn.commit()
print("✅ All fixes committed!")

# 8. Verification
print("\n" + "="*50)
print("VERIFICATION")
print("="*50)

# Count by bank
cur.execute("""
    SELECT b.bank_name, COUNT(*) as count
    FROM reviews r
    JOIN banks b ON r.bank_id = b.bank_id
    GROUP BY b.bank_name
    ORDER BY count DESC;
""")

print("📊 Reviews per bank:")
for bank_name, count in cur.fetchall():
    print(f"   {bank_name}: {count} reviews")

# Average rating
cur.execute("""
    SELECT b.bank_name, ROUND(AVG(r.rating), 2) as avg_rating
    FROM reviews r
    JOIN banks b ON r.bank_id = b.bank_id
    WHERE r.rating IS NOT NULL
    GROUP BY b.bank_name
    ORDER BY avg_rating DESC;
""")

print("\n⭐ Average rating per bank:")
for bank_name, avg_rating in cur.fetchall():
    print(f"   {bank_name}: {avg_rating}")

# Data completeness
cur.execute("""
    SELECT 
        COUNT(*) as total,
        COUNT(bank_id) as with_bank,
        COUNT(rating) as with_rating,
        COUNT(review_date) as with_date,
        COUNT(source) as with_source
    FROM reviews;
""")

total, with_bank, with_rating, with_date, with_source = cur.fetchone()

print(f"\n✅ Data Completeness:")
print(f"   Bank ID: {with_bank}/{total} ({with_bank/total*100:.1f}%)")
print(f"   Rating: {with_rating}/{total} ({with_rating/total*100:.1f}%)")
print(f"   Date: {with_date}/{total} ({with_date/total*100:.1f}%)")
print(f"   Source: {with_source}/{total} ({with_source/total*100:.1f}%)")

# Close connection
cur.close()
conn.close()

print("\n" + "="*50)
print("🎉 DATABASE FIXED AND READY FOR TASK 3!")
print("="*50)

🔄 FINAL DATA FIX SCRIPT
📊 Total reviews to fix: 1200
🔗 Fixing bank_id assignments...
   Updated 100 reviews...
   Updated 200 reviews...
   Updated 300 reviews...
   Updated 400 reviews...
   Updated 500 reviews...
   Updated 600 reviews...
   Updated 700 reviews...
   Updated 800 reviews...
   Updated 900 reviews...
   Updated 1000 reviews...
   Updated 1100 reviews...
   Updated 1200 reviews...
⭐ Fixing ratings...
📅 Fixing review dates...
🌐 Fixing sources...
✅ All fixes committed!

VERIFICATION
📊 Reviews per bank:
   Dashen Bank: 400 reviews
   Bank of Abyssinia: 400 reviews
   Commercial Bank of Ethiopia: 400 reviews

⭐ Average rating per bank:
   Bank of Abyssinia: 3.98
   Dashen Bank: 3.77
   Commercial Bank of Ethiopia: 3.72

✅ Data Completeness:
   Bank ID: 1200/1200 (100.0%)
   Rating: 1200/1200 (100.0%)
   Date: 1200/1200 (100.0%)
   Source: 1200/1200 (100.0%)

🎉 DATABASE FIXED AND READY FOR TASK 3!


In [2]:
# ==============================================
# COMPLETE DATA FIX SCRIPT
# ==============================================

import pandas as pd
import psycopg2
import numpy as np
from datetime import datetime

print("🔧 COMPLETE DATA FIX SCRIPT")
print("="*60)

# Load your actual data
print("📂 Loading your CSV data...")
df_banks_full = pd.read_csv("data/processed/bank_reviews_cleaned.csv")
df_reviews_sentiment = pd.read_csv("data/processed/reviews_for_database.csv")

print(f"   • Loaded {len(df_banks_full)} rows from bank_reviews_cleaned.csv")
print(f"   • Loaded {len(df_reviews_sentiment)} rows from reviews_for_database.csv")

# Connect to PostgreSQL
conn = psycopg2.connect(
    dbname="bank_reviews",
    user="postgres",
    host="localhost",
    port="5432"
)
conn.autocommit = False
cur = conn.cursor()

try:
    print("\n🔄 Step 1: Clearing existing data...")
    cur.execute("DELETE FROM reviews;")
    print("   ✅ Cleared all reviews")
    
    print("\n🔄 Step 2: Ensuring banks table is correct...")
    # Get unique banks from CSV
    unique_banks = df_banks_full[['bank_code', 'bank']].drop_duplicates()
    
    for _, row in unique_banks.iterrows():
        cur.execute("""
            INSERT INTO banks (bank_id, bank_name, app_name)
            VALUES (%s, %s, %s)
            ON CONFLICT (bank_id) DO UPDATE SET
                bank_name = EXCLUDED.bank_name;
        """, (row['bank_code'], row['bank'], None))
    print(f"   ✅ Ensured {len(unique_banks)} banks exist")
    
    print("\n🔄 Step 3: Preparing review data...")
    # Merge all review data
    df_reviews_combined = df_banks_full.merge(
        df_reviews_sentiment[['review_id', 'sentiment_label', 'sentiment_score']],
        on='review_id',
        how='left'
    )
    
    # Select and rename columns
    df_reviews_final = df_reviews_combined[[
        'review_id', 'bank_code', 'review_text', 'rating', 
        'date', 'sentiment_label', 'sentiment_score', 'source'
    ]].copy()
    
    df_reviews_final = df_reviews_final.rename(columns={
        'bank_code': 'bank_id',
        'date': 'review_date'
    })
    
    # Convert types
    df_reviews_final['rating'] = pd.to_numeric(df_reviews_final['rating'], errors='coerce')
    df_reviews_final['review_date'] = pd.to_datetime(df_reviews_final['review_date'], errors='coerce')
    
    print(f"   ✅ Prepared {len(df_reviews_final)} reviews for insertion")
    
    print("\n🔄 Step 4: Inserting all reviews...")
    inserted = 0
    for _, row in df_reviews_final.iterrows():
        # Prepare date
        review_date = row['review_date']
        if pd.isna(review_date):
            review_date = None
        elif hasattr(review_date, 'date'):
            review_date = review_date.date()
        
        # Insert
        cur.execute("""
            INSERT INTO reviews (review_id, bank_id, review_text, rating, 
                               review_date, sentiment_label, sentiment_score, source)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            row['review_id'],
            row['bank_id'],
            row['review_text'][:5000],  # Limit text length
            row['rating'] if not pd.isna(row['rating']) else None,
            review_date,
            row['sentiment_label'],
            row['sentiment_score'],
            row['source']
        ))
        
        inserted += 1
        if inserted % 100 == 0:
            print(f"   Inserted {inserted} reviews...")
    
    conn.commit()
    print(f"\n✅ SUCCESS! Inserted {inserted} reviews with complete data")
    
except Exception as e:
    conn.rollback()
    print(f"\n❌ ERROR: {e}")
    raise
    
finally:
    cur.close()
    conn.close()

print("\n" + "="*60)
print("🎉 DATABASE IS NOW FULLY POPULATED!")
print("="*60)

🔧 COMPLETE DATA FIX SCRIPT
📂 Loading your CSV data...
   • Loaded 1200 rows from bank_reviews_cleaned.csv
   • Loaded 1200 rows from reviews_for_database.csv

🔄 Step 1: Clearing existing data...
   ✅ Cleared all reviews

🔄 Step 2: Ensuring banks table is correct...
   ✅ Ensured 3 banks exist

🔄 Step 3: Preparing review data...
   ✅ Prepared 1200 reviews for insertion

🔄 Step 4: Inserting all reviews...
   Inserted 100 reviews...
   Inserted 200 reviews...
   Inserted 300 reviews...
   Inserted 400 reviews...
   Inserted 500 reviews...
   Inserted 600 reviews...
   Inserted 700 reviews...
   Inserted 800 reviews...
   Inserted 900 reviews...
   Inserted 1000 reviews...
   Inserted 1100 reviews...
   Inserted 1200 reviews...

✅ SUCCESS! Inserted 1200 reviews with complete data

🎉 DATABASE IS NOW FULLY POPULATED!


In [3]:
# ==============================================
# FINAL VERIFICATION FOR GIT COMMIT
# ==============================================

import pandas as pd
import psycopg2

print("✅ FINAL GIT READY VERIFICATION")
print("="*50)

conn = psycopg2.connect(
    dbname="bank_reviews",
    user="postgres",
    host="localhost",
    port="5432"
)

# Check all Task 3 requirements
checks = []

# 1. Database exists
cur = conn.cursor()
cur.execute("SELECT current_database();")
db_name = cur.fetchone()[0]
checks.append(("Database exists", db_name == "bank_reviews"))

# 2. Both tables exist
cur.execute("""
    SELECT COUNT(*) 
    FROM information_schema.tables 
    WHERE table_schema = 'public' 
    AND table_name IN ('banks', 'reviews');
""")
table_count = cur.fetchone()[0]
checks.append(("Both tables exist", table_count == 2))

# 3. Banks table has data
cur.execute("SELECT COUNT(*) FROM banks;")
bank_count = cur.fetchone()[0]
checks.append(("Banks table populated", bank_count >= 3))

# 4. Reviews table has >1000 entries
cur.execute("SELECT COUNT(*) FROM reviews;")
review_count = cur.fetchone()[0]
checks.append((f"Reviews > 1000 entries", review_count > 1000))

# 5. Reviews have bank_id (foreign key)
cur.execute("SELECT COUNT(*) FROM reviews WHERE bank_id IS NULL;")
null_bank_id = cur.fetchone()[0]
checks.append(("All reviews linked to banks", null_bank_id == 0))

# 6. At least 400 reviews have data
cur.execute("""
    SELECT COUNT(*) 
    FROM reviews 
    WHERE bank_id IS NOT NULL 
    AND rating IS NOT NULL;
""")
valid_reviews = cur.fetchone()[0]
checks.append((f"At least 400 valid reviews", valid_reviews >= 400))

# 7. Foreign key constraint exists
cur.execute("""
    SELECT COUNT(*)
    FROM information_schema.table_constraints tc
    WHERE tc.table_name = 'reviews' 
    AND tc.constraint_type = 'FOREIGN KEY';
""")
fk_exists = cur.fetchone()[0]
checks.append(("Foreign key constraint exists", fk_exists > 0))

cur.close()

# Print results
print("\n📋 TASK 3 REQUIREMENTS CHECK:")
print("-" * 40)
all_passed = True
for check_name, passed in checks:
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status}: {check_name}")
    if not passed:
        all_passed = False

# Final summary
print("\n" + "="*50)
if all_passed:
    print("🎉 ALL TASK 3 REQUIREMENTS MET!")
    print("Ready to commit to GitHub!")
else:
    print("⚠️  Some requirements not met.")
    print("Run the fix script above before committing.")

# Show final counts
print("\n📊 FINAL COUNTS:")
df_summary = pd.read_sql("""
    SELECT 
        (SELECT COUNT(*) FROM banks) as banks_count,
        (SELECT COUNT(*) FROM reviews) as reviews_count,
        (SELECT COUNT(DISTINCT bank_id) FROM reviews) as banks_with_reviews,
        (SELECT ROUND(AVG(rating), 2) FROM reviews WHERE rating IS NOT NULL) as avg_rating
""", conn)
print(df_summary)

conn.close()

print("\n" + "="*50)
print("📁 Files ready for GitHub commit:")
print("   1. db_insert_bank_reviews.ipynb (this notebook)")
print("   2. schema.sql (exported schema)")
print("   3. README.md (documentation)")

✅ FINAL GIT READY VERIFICATION

📋 TASK 3 REQUIREMENTS CHECK:
----------------------------------------
✅ PASS: Database exists
✅ PASS: Both tables exist
✅ PASS: Banks table populated
✅ PASS: Reviews > 1000 entries
✅ PASS: All reviews linked to banks
✅ PASS: At least 400 valid reviews
✅ PASS: Foreign key constraint exists

🎉 ALL TASK 3 REQUIREMENTS MET!
Ready to commit to GitHub!

📊 FINAL COUNTS:
   banks_count  reviews_count  banks_with_reviews  avg_rating
0            3           1200                   3        3.82

📁 Files ready for GitHub commit:
   1. db_insert_bank_reviews.ipynb (this notebook)
   2. schema.sql (exported schema)
   3. README.md (documentation)


C:\Users\user\AppData\Local\Temp\ipykernel_28456\1929230371.py:95: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_summary = pd.read_sql("""


In [ ]:
# ==============================================
# EXPORT SCHEMA FOR GITHUB
# ==============================================

import subprocess
import os

print("📁 Exporting database schema...")

# Create schema.sql file
schema_content = """
-- PostgreSQL Schema for Bank Reviews Database
-- Generated for Task 3: Store Cleaned Data in PostgreSQL

-- Database: bank_reviews

-- Table: banks
CREATE TABLE IF NOT EXISTS banks (
    bank_id VARCHAR(10) PRIMARY KEY,
    bank_name VARCHAR(100) NOT NULL,
    app_name VARCHAR(100)
);

-- Table: reviews
CREATE TABLE IF NOT EXISTS reviews (
    review_id VARCHAR(50) PRIMARY KEY,
    bank_id VARCHAR(10) REFERENCES banks(bank_id),
    review_text TEXT NOT NULL,
    rating NUMERIC(3,1),
    review_date DATE,
    sentiment_label VARCHAR(20),
    sentiment_score NUMERIC(3,2),
    source VARCHAR(50)
);

-- Indexes for better query performance
CREATE INDEX IF NOT EXISTS idx_reviews_bank_id ON reviews(bank_id);
CREATE INDEX IF NOT EXISTS idx_reviews_sentiment ON reviews(sentiment_label);
CREATE INDEX IF NOT EXISTS idx_reviews_date ON reviews(review_date);

-- Sample data for banks table
INSERT INTO banks (bank_id, bank_name, app_name) VALUES
('BOA', 'Bank of Abyssinia', 'BOA Mobile Banking'),
('CBE', 'Commercial Bank of Ethiopia', 'CBE Birr'),
('DASHEN', 'Dashen Bank', 'Dashen Bank Mobile')
ON CONFLICT (bank_id) DO NOTHING;

-- Verification queries (for documentation)
/*
-- Count reviews per bank
SELECT b.bank_name, COUNT(r.review_id) as review_count
FROM reviews r
JOIN banks b ON r.bank_id = b.bank_id
GROUP BY b.bank_name
ORDER BY review_count DESC;

-- Average rating per bank
SELECT b.bank_name, ROUND(AVG(r.rating), 2) as avg_rating
FROM reviews r
JOIN banks b ON r.bank_id = b.bank_id
WHERE r.rating IS NOT NULL
GROUP BY b.bank_name
ORDER BY avg_rating DESC;
*/
"""

# Save to file
with open('schema.sql', 'w', encoding='utf-8') as f:
    f.write(schema_content)

print("✅ schema.sql created successfully!")

# Also create a simple README.md
readme_content = """# Bank Reviews Database

## Task 3: Store Cleaned Data in PostgreSQL

### Database Schema

#### Table: `banks`
- `bank_id` (VARCHAR, PRIMARY KEY): Unique identifier for each bank
- `bank_name` (VARCHAR): Full name of the bank
- `app_name` (VARCHAR): Name of the mobile banking application

#### Table: `reviews`
- `review_id` (VARCHAR, PRIMARY KEY): Unique identifier for each review
- `bank_id` (VARCHAR, FOREIGN KEY): Reference to banks table
- `review_text` (TEXT): The actual review content
- `rating` (NUMERIC): User rating (1-5 scale)
- `review_date` (DATE): Date when review was posted
- `sentiment_label` (VARCHAR): Sentiment classification (POSITIVE/NEGATIVE/NEUTRAL)
- `sentiment_score` (NUMERIC): Sentiment analysis confidence score (0-1)
- `source` (VARCHAR): Source of the review (e.g., Google Play Store)

### Key Features
- **Referential Integrity**: Foreign key constraint ensures all reviews reference valid banks
- **Data Validation**: Appropriate data types and constraints
- **Performance**: Indexes on frequently queried columns
- **Scalability**: Supports large volumes of review data

### Current Statistics
- **Banks**: 3 (Bank of Abyssinia, Commercial Bank of Ethiopia, Dashen Bank)
- **Total Reviews**: 1,200+
- **Average Rating**: 3.82/5
- **Data Completeness**: 100% bank linkage, minimal NULL values

### Usage
1. Create database: `createdb bank_reviews`
2. Import schema: `psql -d bank_reviews -f schema.sql`
3. Run the Jupyter notebook `db_insert_bank_reviews.ipynb` to populate data

### Verification Queries
See `schema.sql` for sample verification queries.

### Files in Repository
- `db_insert_bank_reviews.ipynb`: Complete Jupyter notebook with all code
- `schema.sql`: Database schema and sample data
- `README.md`: This documentation file
"""

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)

print("✅ README.md created successfully!")

print("\n" + "="*50)
print("📁 FILES READY FOR GITHUB:")
print("   1. schema.sql - Database schema")
print("   2. README.md - Documentation")
print("   3. db_insert_bank_reviews.ipynb - Your notebook")
print("="*50)